In [ ]:
"""

Data Assimilation Preprocessing

Workflow
1. Station data
    1a. Joins station data csvs with the metadata csv to bring in elevation, lat/long associated with each station id 
    1b. Collapse station data to the hourly level by... at each target hour, collect all station observations within the hour and 
    average for that station
    1c. Generally the station csvs contain predictors for temp_air, temp_dew, and rh. For any predictors that were missing 
    before (i.e., NA), calculate them using foundational equations found in model_meteo(). 
    Calculate temp_bulb based on equations found in model_meteo(). 
2. IMERG: 
    2a. Convert wide to long and average half hourly data to the hourly level. 
3. MRoS: 
    3a. There might be multiple observations coming from the same observer within an hour. 
    If that's the case, choose the latter observation that was recorded (i.e., if an observer changed their mind about the phase). 
    Otherwise, floor each MRoS observation datetime_UTC to the starting hour. 
4. At this point, all the data should have lat, lon, datetime_utc (hourly level), predictors. 
    Filter all of them to the lat/long within our DEM AOI. 

"""

import re
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box
import rasterio as rio
from rasterio.warp import transform_bounds, reproject, Resampling, calculate_default_transform
from pyproj import CRS, Transformer
import pytz


In [ ]:
# --------------------------- CONFIG ---------------------------------
BASE_DIR = Path().resolve().parent  # current working dir
print("BASE_DIR:", BASE_DIR)


CONFIG = {
    "wy_start": "2024-10-01T00:00:00Z",
    "wy_end":   "2025-05-31T23:59:59Z",
    # "test_start": "2025-02-01T00:00:00Z",   # narrow test window first
    # "test_end":   "2025-04-01T00:00:00Z",
    "test_start": "2024-10-01T00:00:00Z",   # Entire window
    "test_end":   "2025-05-31T23:59:59Z",

    "station_meta_csv": BASE_DIR / "Data/Stations/station_metadata_20241001_20250531.csv",
    "station_dir": BASE_DIR / "Data/Stations",   # per-station CSVs, temp data is already converted to Celcius via preprocess_meteo() from rainorsnowtools
    "imerg_dir":   BASE_DIR / "Data/IMERG",      # parquet (wide)
    "mros_parquet": BASE_DIR / "Data/observations/wy25_mros_obs.parquet",

    "dem_path": "C:/Users/EmmaGolub/Desktop/MRoS_local/local_data/DEM_AOI_TNM_10m.tif",
    "dem_path_km": BASE_DIR / "DEM_1km.tif",
    "out_dir":  BASE_DIR / "outputs/hourly_pipeline",
}

out_dir = Path(CONFIG["out_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

BASE_DIR: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype


In [ ]:
# ------------------------- UTIL: time --------------------------------
def to_utc(dt_series: pd.Series) -> pd.DatetimeIndex:
    """Force timestamps to UTC, making naive → UTC-naive assumed in UTC."""
    dt = pd.to_datetime(dt_series, errors="coerce", utc=True)
    # If dt_series had naive datetimes and pandas assumed local, .tz_convert('UTC') not needed.
    return dt

def hourly_index(start_iso: str, end_iso: str) -> pd.DatetimeIndex:
    return pd.date_range(start=pd.to_datetime(start_iso), end=pd.to_datetime(end_iso),
                         freq="H", tz="UTC")

def print_time(ts):
    return pd.to_datetime(ts).strftime("%Y-%m-%d %H:%MZ")

In [ ]:
# ---------------------- UTIL: meteorology functions -------------------------
# Magnus (Tetens) saturation vapor pressure over water (°C)
def esat_hpa(Tc: float) -> float:
    return 6.112 * np.exp(17.67 * Tc / (Tc + 243.5))

def td_from_ta_rh(TaC: np.ndarray, RH: np.ndarray) -> np.ndarray:
    """Dewpoint from air temp (°C) and RH (%)"""
    TaC = np.asarray(TaC, dtype=float)
    RH = np.clip(np.asarray(RH, dtype=float), 1e-6, 100.0)
    a, b = 17.625, 243.04
    gamma = np.log(RH / 100.0) + (a * TaC) / (b + TaC)
    Td = (b * gamma) / (a - gamma)
    return Td

def rh_from_ta_td(TaC: np.ndarray, TdC: np.ndarray) -> np.ndarray:
    """RH (%) from air temp and dewpoint (°C)"""
    TaC = np.asarray(TaC, dtype=float)
    TdC = np.asarray(TdC, dtype=float)
    a, b = 17.625, 243.04
    ln_es_Ta = (a * TaC) / (b + TaC)
    ln_es_Td = (a * TdC) / (b + TdC)
    RH = 100.0 * np.exp(ln_es_Td - ln_es_Ta)
    return np.clip(RH, 0.0, 100.0)

def tw_stull(TaC: np.ndarray, RH: np.ndarray) -> np.ndarray:
    """
    Wet-bulb approximation (°C) by Stull (2011).
    TaC in °C, RH in %
    """
    TaC = np.asarray(TaC, dtype=float)
    RH = np.clip(np.asarray(RH, dtype=float), 1e-6, 100.0)
    Tw = (TaC * np.arctan(0.151977 * np.sqrt(RH + 8.313659)) +
          np.arctan(TaC + RH) - np.arctan(RH - 1.676331) +
          0.00391838 * RH**1.5 * np.arctan(0.023101 * RH) - 4.686035)
    return Tw

def fill_station_row_vars(df: pd.DataFrame) -> pd.DataFrame:
    """
      - If RH is NaN and Ta & Td present -> compute RH.
      - If Td is NaN and Ta & RH present -> compute Td.
      - Clamp RH to [0,100].
      - If Ta & RH present -> compute Tw.
    Assumes temps in °C and RH in %.
    """

    df = df.copy()

    # Coerce to numeric
    for col in ["temp_air", "temp_dew", "rh"]:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Track NA counts before
    na_before = df[["temp_air", "temp_dew", "rh"]].isna().sum()
    tw_before = df.get("temp_wet").isna().sum() if "temp_wet" in df else df.shape[0]

    # Compute RH when Ta & Td present but RH missing
    mask_rh = df["rh"].isna() & df["temp_air"].notna() & df["temp_dew"].notna()
    if mask_rh.any():
        df.loc[mask_rh, "rh"] = rh_from_ta_td(
            df.loc[mask_rh, "temp_air"].astype(float).values,
            df.loc[mask_rh, "temp_dew"].astype(float).values,
        )

    # Compute Td when Ta & RH present but Td missing
    mask_td = df["temp_dew"].isna() & df["temp_air"].notna() & df["rh"].notna()
    if mask_td.any():
        df.loc[mask_td, "temp_dew"] = td_from_ta_rh(
            df.loc[mask_td, "temp_air"].astype(float).values,
            df.loc[mask_td, "rh"].astype(float).values,
        )

    # Clamp RH and compute Tw where possible
    if "rh" in df:
        df["rh"] = df["rh"].clip(0.0, 100.0)

    mask_tw = df["temp_air"].notna() & df["rh"].notna()
    if mask_tw.any():
        df.loc[mask_tw, "temp_wet"] = tw_stull(
            df.loc[mask_tw, "temp_air"].astype(float).values,
            df.loc[mask_tw, "rh"].astype(float).values,
        )

    # Report how many were filled
    na_after = df[["temp_air", "temp_dew", "rh"]].isna().sum()
    tw_after = df["temp_wet"].isna().sum() if "temp_wet" in df else df.shape[0]
    filled_rh = int((na_before["rh"] - na_after["rh"]))
    filled_td = int((na_before["temp_dew"] - na_after["temp_dew"]))
    filled_tw = int((tw_before - tw_after))
    total = len(df)
    print(
        f"[supplement] Filled RH: {filled_rh} ({filled_rh/total:.1%}), "
        f"Td: {filled_td} ({filled_td/total:.1%}), "
        f"Tw: {filled_tw} ({filled_tw/total:.1%})"
    )

    return df



In [ ]:
# # -------------------- LOAD: DEM & AOI functions ------------------------

def load_dem_and_aoi(dem_path: str):
    with rio.open(dem_path) as src:
        dem_crs = CRS.from_wkt(src.crs.to_wkt()) if src.crs else None
        bounds = src.bounds
        aoi_wgs84 = transform_bounds(src.crs, "EPSG:4326",
                                     bounds.left, bounds.bottom, bounds.right, bounds.top,
                                     densify_pts=21)
    aoi_poly = box(aoi_wgs84[0], aoi_wgs84[1], aoi_wgs84[2], aoi_wgs84[3])
    return dem_path, dem_crs, aoi_poly

# def reproject_resample_dem_to_1km(dem_path: str, proj_fallback="EPSG:3310"):
#     with rio.open(dem_path) as src:
#         src_crs = CRS.from_wkt(src.crs.to_wkt()) if src.crs else None
#         if not src_crs or src_crs.is_geographic:
#             dst_crs = CRS.from_string(proj_fallback)
#             transform, width, height = calculate_default_transform(
#                 src.crs, dst_crs, src.width, src.height, *src.bounds
#             )
#             profile = src.profile.copy()
#             profile.update(crs=dst_crs, transform=transform, width=width, height=height)
#             data_proj = np.empty((height, width), dtype="float32")
#             reproject(
#                 source=rio.band(src, 1),
#                 destination=data_proj,
#                 src_transform=src.transform,
#                 src_crs=src.crs,
#                 dst_transform=transform,
#                 dst_crs=dst_crs,
#                 resampling=Resampling.bilinear
#             )
#             dem_proj, profile = data_proj, profile
#         else:
#             dem_proj = src.read(1).astype("float32")
#             profile = src.profile.copy()

#     # Resample to ~1 km via average
#     xres = abs(profile["transform"].a)
#     yres = abs(profile["transform"].e)
#     target_res = 1000.0
#     new_w = max(1, int(np.floor(profile["width"]  * (xres/target_res))))
#     new_h = max(1, int(np.floor(profile["height"] * (abs(yres)/target_res))))

#     dst = np.empty((new_h, new_w), dtype=np.float32)
#     dst_transform = rio.Affine(
#         target_res, 0.0, profile["transform"].c,
#         0.0, -target_res, profile["transform"].f
#     )
#     reproject(
#         source=dem_proj,
#         destination=dst,
#         src_transform=profile["transform"],
#         src_crs=profile["crs"],
#         dst_transform=dst_transform,
#         dst_crs=profile["crs"],
#         resampling=Resampling.average
#     )
#     out_prof = profile.copy()
#     out_prof.update({"height": new_h, "width": new_w, "transform": dst_transform,
#                      "dtype":"float32", "count":1})
#     return dst, out_prof

def grid_centers(profile):
    T = profile["transform"]
    xs = T.c + (np.arange(profile["width"]) + 0.5) * T.a
    ys = T.f + (np.arange(profile["height"]) + 0.5) * T.e
    X, Y = np.meshgrid(xs, ys)
    return np.column_stack([X.ravel(), Y.ravel()])

# Load DEM/AOI now
dem_path, dem_crs, aoi_poly = load_dem_and_aoi(CONFIG["dem_path"])
# dem1k_data, dem1k_profile = reproject_resample_dem_to_1km(dem_path, CONFIG["proj_fallback"])
# grid_xy = grid_centers(dem1k_profile)
# grid_elev = dem1k_data.ravel()
# proj_crs = dem1k_profile["crs"]

# print(f"DEM 1-km grid: {dem1k_profile['width']} x {dem1k_profile['height']} | "
#       f"res ≈ {abs(dem1k_profile['transform'].a)} m")

In [ ]:
# # Save resampled DEM to GeoTIFF
# out_dem_path = out_dir / "DEM_1km.tif"

# with rio.open(out_dem_path, "w", **dem1k_profile) as dst:
#     dst.write(dem1k_data, 1)

# print(f"Saved 1-km DEM to {out_dem_path}")

In [ ]:
# Load already projected and saved 1km DEM tif

with rio.open(CONFIG["dem_path_km"]) as src:
    dem1k_profile = src.profile   # metadata
    dem1k_data = src.read(1)      # pixel values

    # Optional extras
    dem_crs = src.crs             # CRS object
    dem_bounds = src.bounds       # bounding box
    dem_transform = src.transform # affine transform

grid_xy = grid_centers(dem1k_profile)
grid_elev = dem1k_data.ravel()
proj_crs = dem1k_profile["crs"]

print(f"DEM 1-km grid: {dem1k_profile['width']} x {dem1k_profile['height']} | "
      f"res ≈ {abs(dem1k_profile['transform'].a)} m")

# Function for appending elevation from this DEM to a dataset (i.e., for IMERG, MRoS...)
def add_elev_from_src(df, src, lon_col="lon", lat_col="lat"):
    coords = list(zip(df[lon_col], df[lat_col]))
    elev = np.array([val[0] for val in src.sample(coords)])
    return df.assign(elev=elev)

In [ ]:
# -------------------- LOAD: Stations ---------------------------------

def load_station_meta(meta_csv: str) -> pd.DataFrame:
    meta = pd.read_csv(meta_csv)
    req = {"id","lat","lon","elev","timezone_lst"}
    missing = req - set(meta.columns)
    if missing:
        raise ValueError(f"Station metadata missing columns: {missing}")
    meta["id"] = meta["id"].astype(str)
    return meta

def load_station_timeseries(station_dir: str, meta: pd.DataFrame) -> pd.DataFrame:
    files = [p for p in Path(station_dir).glob("*.csv") if "meta" not in p.name.lower()]
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        if "id" not in df.columns:
            df["id"] = f.stem
        keep = ["id","datetime","temp_air","temp_dew","rh"]
        for k in keep:
            if k not in df.columns:
                df[k] = np.nan
        df = df[keep]
        df["id"] = df["id"].astype(str)

        # timezone per station
        tz_vals = meta.loc[meta["id"] == df["id"].iloc[0], "timezone_lst"].values
        dt_local = pd.to_datetime(df["datetime"], errors="coerce")
        if len(tz_vals) == 1:
            try:
                tz = pytz.timezone(tz_vals[0])
                if getattr(dt_local.dt, "tz", None) is None:
                    df["datetime"] = dt_local.dt.tz_localize(tz, ambiguous="NaT", nonexistent="NaT").dt.tz_convert("UTC")
                else:
                    df["datetime"] = dt_local.dt.tz_convert("UTC")
            except Exception as e:
                print(f"Warning: timezone '{tz_vals}' failed for station {df['id'].iloc[0]}: {e}")
                df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", utc=True)
        else:
            df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", utc=True)
        dfs.append(df)
    if not dfs:
        return pd.DataFrame(columns=["id","datetime","temp_air","temp_dew","rh"])
    return pd.concat(dfs, ignore_index=True)

def hourly_station_agg(st_df: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    df = st_df.merge(meta, on="id", how="left")
    df["hour_utc"] = df["datetime"].dt.floor("H")
    agg = (df.groupby(["id","hour_utc"], as_index=False)
             .agg(temp_air=("temp_air","mean"),
                  temp_dew=("temp_dew","mean"),
                  rh=("rh","mean"),
                  lat=("lat","first"),
                  lon=("lon","first"),
                  elev=("elev","first")))
    agg = fill_station_row_vars(agg)

    # ---- CLEAN TEMPERATURE OUTLIERS  ----
    cutoffs = {
        "temp_air": (-80, 60), # thresholds of reasonable Celcius values
        "temp_dew": (-80, 40),
        "temp_wet": (-80, 50),
    }
    for col, (lo, hi) in cutoffs.items():
        if col in agg.columns:
            agg.loc[(agg[col] < lo) | (agg[col] > hi), col] = np.nan

    # consistency checks
    if "temp_air" in agg.columns and "temp_dew" in agg.columns:
        agg.loc[agg["temp_dew"] > agg["temp_air"], "temp_dew"] = np.nan
    if "temp_air" in agg.columns and "temp_wet" in agg.columns:
        agg.loc[agg["temp_wet"] > agg["temp_air"], "temp_wet"] = np.nan

    return agg

# run
meta = load_station_meta(CONFIG["station_meta_csv"])
st_ts = load_station_timeseries(CONFIG["station_dir"], meta)
st_hr = hourly_station_agg(st_ts, meta)
# window filter
st_hr = st_hr[(st_hr["hour_utc"] >= pd.to_datetime(CONFIG["test_start"])) &
              (st_hr["hour_utc"] <= pd.to_datetime(CONFIG["test_end"]))]

print(f"Stations hourly rows in window: {len(st_hr)}")

In [ ]:
# -------------------- LOAD: IMERG (wide→long→hourly) -----------------

# Dates like "2024-10-11 01:30:00" (space or "T" also OK)
TIME_COL_RE = re.compile(r"^\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}(?::\d{2})?$")
DATE_ONLY_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")

def read_imerg_wide_to_long(path: Path) -> pd.DataFrame:
    import pyarrow.parquet as pq
    table = pq.read_table(path)
    df = table.to_pandas()
    df = df.rename(columns={"x": "lon", "y": "lat"})

    # Only use the columns that include a time; drop pure date columns
    time_cols = [c for c in df.columns if TIME_COL_RE.match(str(c))]
    if not time_cols:
        # fall back: if nothing matched, try any time-like and then remove date-only
        any_timeish = [c for c in df.columns if re.match(r"^\d{4}-\d{2}-\d{2}", str(c))]
        time_cols = [c for c in any_timeish if not DATE_ONLY_RE.match(str(c))]
    if not time_cols:
        raise ValueError(f"No time-like (HH:MM) columns in {path}")

    long = df.melt(
        id_vars=["lat", "lon"],
        value_vars=time_cols,
        var_name="time_str",
        value_name="plp_raw",
    )

    # Parse timestamps as UTC; these already include hour/minute
    long["time_utc"] = pd.to_datetime(long["time_str"], utc=True, errors="coerce")

    # Values → float; scale to percent if the data are 0–1
    plp = pd.to_numeric(long["plp_raw"], errors="coerce").astype(float)
    if np.nanmax(plp) <= 1.0:
        plp *= 100.0
    long["plp"] = plp

    return long.loc[long["time_utc"].notna(), ["time_utc", "lat", "lon", "plp"]]

def hourly_imerg(imerg_dir: str, start_iso: str, end_iso: str) -> pd.DataFrame:
    files = list(Path(imerg_dir).rglob("*.parquet"))
    if not files:
        print(f"[IMERG] No parquet files under {imerg_dir}")
        return pd.DataFrame(columns=["hour_utc", "lat", "lon", "plp"])

    start_ts = pd.to_datetime(start_iso, utc=True)
    end_ts   = pd.to_datetime(end_iso,   utc=True)

    dfs = []
    for f in files:
        df = read_imerg_wide_to_long(f)
        df = df[(df["time_utc"] >= start_ts) & (df["time_utc"] <= end_ts)]
        if not df.empty:
            dfs.append(df)

    if not dfs:
        print("[IMERG] Found files but no rows within the requested window.")
        return pd.DataFrame(columns=["hour_utc", "lat", "lon", "plp"])

    imerg = pd.concat(dfs, ignore_index=True)

    # Average the two half-hour slots within each hour per pixel
    imerg["hour_utc"] = imerg["time_utc"].dt.floor("h")
    imerg_hr = (imerg.groupby(["hour_utc", "lat", "lon"], as_index=False)
                      .agg(plp=("plp", "mean")))
    return imerg_hr

# run
imerg_hr = hourly_imerg(CONFIG["imerg_dir"], CONFIG["test_start"], CONFIG["test_end"])

# Append elevation
imerg_hr = add_elev_from_src(imerg_hr, src)

print(f"IMERG hourly points in window: {len(imerg_hr)}")

In [ ]:
# -------------------- LOAD: MRoS -------------------------------------
def load_mros(mros_parquet: str, start_iso: str, end_iso: str) -> pd.DataFrame:
    import pyarrow.parquet as pq
    table = pq.read_table(mros_parquet)
    df = table.to_pandas()

    if "datetime_utc" not in df.columns:
        dt = pd.to_datetime(df["date_submitted_utc"] + " " + df["time_submitted_utc"],
                            utc=True, errors="coerce")
        df["datetime_utc"] = dt
    df["hour_utc"] = df["datetime_utc"].dt.floor("H")
    df["phase"] = df["phase"].str.lower()

    key_cols = ["hour_utc"]
    if "observer_id" in df.columns:
        key_cols.append("observer_id")
    else:
        df["lat_bin"] = pd.to_numeric(df["latitude"], errors="coerce").round(4)
        df["lon_bin"] = pd.to_numeric(df["longitude"], errors="coerce").round(4)
        key_cols += ["lat_bin","lon_bin"]

    df = df.sort_values("datetime_utc")
    last = df.groupby(key_cols, as_index=False).tail(1)

    map_plp = {"snow":0.0, "mix":50.0, "rain":100.0}
    last["mros_plp_proxy"] = last["phase"].map(map_plp).astype(float)

    last = last[(last["hour_utc"] >= pd.to_datetime(start_iso)) &
                (last["hour_utc"] <= pd.to_datetime(end_iso))]
    return last.rename(columns={"latitude":"lat","longitude":"lon"})[
        ["hour_utc","lat","lon","mros_plp_proxy","phase"]
    ]

# run
mros_hr = load_mros(CONFIG["mros_parquet"], CONFIG["test_start"], CONFIG["test_end"])

# Append elevation
mros_hr = add_elev_from_src(mros_hr, src)

print(f"MRoS hourly rows in window: {len(mros_hr)}")

In [ ]:
# -------------------- AOI filter -------------------------------------
def filter_points_to_aoi(df: pd.DataFrame, aoi_poly) -> pd.DataFrame:
    g = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326")
    poly = gpd.GeoSeries([aoi_poly], crs="EPSG:4326").iloc[0]
    mask = g.intersects(poly)
    return df.loc[mask.values].drop(columns=["geometry"], errors="ignore")

st_hr   = filter_points_to_aoi(st_hr,   aoi_poly)
imerg_hr= filter_points_to_aoi(imerg_hr,aoi_poly)
mros_hr    = filter_points_to_aoi(mros_hr,    aoi_poly)

print(len(st_hr), len(imerg_hr), len(mros_hr))

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

# Convert DataFrame → Arrow Table → Parquet
pq.write_table(pa.Table.from_pandas(st_hr), out_dir / "stations_hourly.parquet")
pq.write_table(pa.Table.from_pandas(imerg_hr), out_dir / "imerg_hourly.parquet")
pq.write_table(pa.Table.from_pandas(mros_hr), out_dir / "mros_hourly.parquet")

In [ ]:
# How many MRoS reports per hour?
mros_counts = mros_hr.groupby("hour_utc").size().rename("n_mros")
print(mros_counts.describe())
print("hours with >=3:", (mros_counts>=3).sum(), " | >=2:", (mros_counts>=2).sum(), " | >=1:", (mros_counts>=1).sum())

h = pd.Timestamp("2024-10-17 21:00:00+00:00")
print(mros_hr.loc[mros_hr["hour_utc"]==h, ["lat","lon","phase","mros_plp_proxy"]].head(10))